# Paired Sentence Engineering — H3/H4/H5 Data

This notebook builds the controlled sentence datasets for H3, H4, and H5.

## Three datasets to produce

| File | Used in | Description |
|------|---------|-------------|
| `paired_sentences.json` | H3, H4 | Matched L/R pairs: same disambiguating info before vs after homonym |
| `garden_path_sentences.json` | H5 | Left context primes wrong sense; right context corrects it |

## Condition definitions

**L (left-biased):** The disambiguating context comes *before* the homonym.  
*"After years of savings at the financial institution, she visited the bank."*

**R (right-biased):** The disambiguating context comes *after* the homonym.  
*"She visited the bank, where the teller processed her savings transfer."*

**Garden-path:** Left context strongly primes sense A; full sentence resolves to sense B.  
*"She hurried to the bank with her deposit slips — only to find it was a muddy riverbank."*

## Design rules
1. Each L/R pair should carry the **same disambiguating information**, just reordered.
2. The homonym must appear in the sentence as a **single unambiguous token** (check tokenization).
3. No other homonyms in the same sentence.
4. Garden-path sentences need a **primed_sense** label (what the left context implies) AND a **correct_sense** label (what the full sentence means).
5. Aim for **10 pairs per word per condition** (10 L + 10 R + 5-10 garden-path).

## 1. Review existing H2 sentences for natural L/R classification

In [ ]:
import pandas as pd
from pathlib import Path

df = pd.read_pickle("synthetic_data_h2.pkl")

def context_position(sentence: str, word: str) -> str:
    """
    Heuristic: where does most disambiguating content fall?
    'L' = word appears in second half of sentence
    'R' = word appears in first half
    'MID' = word near centre
    """
    pos = sentence.lower().find(word.lower())
    if pos < 0:
        return '?'
    rel = pos / len(sentence)
    if rel > 0.6:
        return 'L'
    elif rel < 0.4:
        return 'R'
    return 'MID'

print(f"{'Word':<8} {'Sense':<6} {'Pos':>4}  Sentence")
print("-" * 90)
for _, row in df.iterrows():
    word  = row['word']
    sense = row['semantic_group_id']
    label = row['sense_label']
    for sent in row['examples'][:5]:  # show first 5 per sense
        pos = context_position(sent, word)
        print(f"{word:<8} {sense:<6} {pos:>4}  {sent[:75]}")
    print()

## 2. Paired sentences (H3 / H4)

For each word, write matched L/R pairs below.  
Each pair should carry the **same disambiguating phrase**, just repositioned.

**Format per item:**
```python
{"id": "bank_L_01", "sentence": "...", "condition": "L", "sense": 0}
{"id": "bank_R_01", "sentence": "...", "condition": "R", "sense": 0}
```

In [ ]:
# ─── EDIT THIS DICT TO ADD/CHANGE PAIRED SENTENCES ───────────────────────────
#
# Each entry is a list of sentence dicts with keys:
#   id        : unique string ID
#   sentence  : the sentence (must contain the homonym)
#   condition : 'L' or 'R'
#   sense     : 0 or 1 (matches semantic_group_id in synthetic_data_h2.pkl)
#
# L = disambiguating context BEFORE the homonym
# R = disambiguating context AFTER the homonym
#
# Each L item should have a corresponding R item with the same sense
# and approximately the same disambiguating content, just reordered.

PAIRED = {
    "bank": [
        # --- Sense 0: river bank ---
        {"id": "bank_L_01", "sentence": "After wading through the shallow stream, she climbed onto the muddy bank.",
         "condition": "L", "sense": 0},
        {"id": "bank_R_01", "sentence": "She climbed onto the bank, which was muddy from the recent flooding of the stream.",
         "condition": "R", "sense": 0},

        {"id": "bank_L_02", "sentence": "Along the river's edge, tall reeds and willows lined the bank.",
         "condition": "L", "sense": 0},
        {"id": "bank_R_02", "sentence": "The bank was lined with tall reeds and willows that overhung the river's edge.",
         "condition": "R", "sense": 0},

        {"id": "bank_L_03", "sentence": "Eroded by years of floodwaters, the bank crumbled beneath her feet.",
         "condition": "L", "sense": 0},
        {"id": "bank_R_03", "sentence": "The bank crumbled beneath her feet, eroded by years of floodwaters.",
         "condition": "R", "sense": 0},

        {"id": "bank_L_04", "sentence": "Where the fishermen cast their lines each morning, the bank was worn smooth.",
         "condition": "L", "sense": 0},
        {"id": "bank_R_04", "sentence": "The bank was worn smooth where the fishermen cast their lines each morning.",
         "condition": "R", "sense": 0},

        {"id": "bank_L_05", "sentence": "After the spring thaw, the bank was covered in soft green moss.",
         "condition": "L", "sense": 0},
        {"id": "bank_R_05", "sentence": "The bank was covered in soft green moss after the spring thaw.",
         "condition": "R", "sense": 0},

        # --- Sense 1: financial institution ---
        {"id": "bank_L_06", "sentence": "To withdraw her savings and pay the mortgage, she visited the bank.",
         "condition": "L", "sense": 1},
        {"id": "bank_R_06", "sentence": "She visited the bank to withdraw her savings and pay the mortgage.",
         "condition": "R", "sense": 1},

        {"id": "bank_L_07", "sentence": "After receiving a loan rejection letter, he drove back to the bank.",
         "condition": "L", "sense": 1},
        {"id": "bank_R_07", "sentence": "He drove to the bank, where he had just received a loan rejection letter.",
         "condition": "R", "sense": 1},

        {"id": "bank_L_08", "sentence": "With her new debit card and PIN in hand, she walked into the bank.",
         "condition": "L", "sense": 1},
        {"id": "bank_R_08", "sentence": "She walked into the bank and received her new debit card and PIN.",
         "condition": "R", "sense": 1},

        {"id": "bank_L_09", "sentence": "To open a savings account, he made an appointment at the bank.",
         "condition": "L", "sense": 1},
        {"id": "bank_R_09", "sentence": "He made an appointment at the bank to open a savings account.",
         "condition": "R", "sense": 1},

        {"id": "bank_L_10", "sentence": "After queuing for twenty minutes behind other customers, he reached the bank teller.",
         "condition": "L", "sense": 1},
        {"id": "bank_R_10", "sentence": "He reached the bank teller after queuing for twenty minutes behind other customers.",
         "condition": "R", "sense": 1},
    ],

    "bat": [
        # Sense 0: sports equipment
        {"id": "bat_L_01", "sentence": "Gripping it tightly at the crease before the bowler ran in, she raised the bat.",
         "condition": "L", "sense": 0},
        {"id": "bat_R_01", "sentence": "She raised the bat, gripping it tightly at the crease before the bowler ran in.",
         "condition": "R", "sense": 0},

        {"id": "bat_L_02", "sentence": "After inspecting its handle for cracks, the umpire approved the bat.",
         "condition": "L", "sense": 0},
        {"id": "bat_R_02", "sentence": "The umpire approved the bat after inspecting its handle for cracks.",
         "condition": "R", "sense": 0},

        {"id": "bat_L_03", "sentence": "Made from English willow and used at the county level, his bat was valuable.",
         "condition": "L", "sense": 0},
        {"id": "bat_R_03", "sentence": "His bat was valuable, made from English willow and used at the county level.",
         "condition": "R", "sense": 0},

        {"id": "bat_L_04", "sentence": "After hitting a six off the final delivery, he kissed the bat.",
         "condition": "L", "sense": 0},
        {"id": "bat_R_04", "sentence": "He kissed the bat after hitting a six off the final delivery.",
         "condition": "R", "sense": 0},

        {"id": "bat_L_05", "sentence": "Taped around its handle to improve grip, the bat was ready for the match.",
         "condition": "L", "sense": 0},
        {"id": "bat_R_05", "sentence": "The bat was ready for the match, taped around its handle to improve grip.",
         "condition": "R", "sense": 0},

        # Sense 1: flying mammal
        {"id": "bat_L_06", "sentence": "Navigating entirely by echolocation in the darkness of the cave, the bat swooped past.",
         "condition": "L", "sense": 1},
        {"id": "bat_R_06", "sentence": "The bat swooped past, navigating entirely by echolocation in the darkness of the cave.",
         "condition": "R", "sense": 1},

        {"id": "bat_L_07", "sentence": "Hanging upside down from the rafter, wrapped in its leathery wings, the bat slept.",
         "condition": "L", "sense": 1},
        {"id": "bat_R_07", "sentence": "The bat slept, hanging upside down from the rafter and wrapped in its leathery wings.",
         "condition": "R", "sense": 1},

        {"id": "bat_L_08", "sentence": "Darting between the trees at dusk to catch insects, the bat was almost invisible.",
         "condition": "L", "sense": 1},
        {"id": "bat_R_08", "sentence": "The bat was almost invisible, darting between the trees at dusk to catch insects.",
         "condition": "R", "sense": 1},

        {"id": "bat_L_09", "sentence": "Tagged by researchers to track migration routes, the bat was released at sunset.",
         "condition": "L", "sense": 1},
        {"id": "bat_R_09", "sentence": "The bat was released at sunset, tagged by researchers to track migration routes.",
         "condition": "R", "sense": 1},

        {"id": "bat_L_10", "sentence": "Roosting in a colony inside the old limestone cave, the bat emerged at nightfall.",
         "condition": "L", "sense": 1},
        {"id": "bat_R_10", "sentence": "The bat emerged at nightfall, roosting in a colony inside the old limestone cave.",
         "condition": "R", "sense": 1},
    ],

    "bark": [
        # Sense 0: dog sound
        {"id": "bark_L_01", "sentence": "Startled by the stranger at the gate, the dog let out a sudden bark.",
         "condition": "L", "sense": 0},
        {"id": "bark_R_01", "sentence": "The dog let out a sudden bark, startled by the stranger at the gate.",
         "condition": "R", "sense": 0},

        {"id": "bark_L_02", "sentence": "When the postman rang the bell, the guard dog's bark echoed through the yard.",
         "condition": "L", "sense": 0},
        {"id": "bark_R_02", "sentence": "The guard dog's bark echoed through the yard when the postman rang the bell.",
         "condition": "R", "sense": 0},

        {"id": "bark_L_03", "sentence": "Trained to alert her owner to intruders, her bark was sharp and controlled.",
         "condition": "L", "sense": 0},
        {"id": "bark_R_03", "sentence": "Her bark was sharp and controlled, trained to alert her owner to intruders.",
         "condition": "R", "sense": 0},

        {"id": "bark_L_04", "sentence": "Soft and playful when greeting his owner, the puppy's bark was barely a sound.",
         "condition": "L", "sense": 0},
        {"id": "bark_R_04", "sentence": "The puppy's bark was barely a sound, soft and playful when greeting his owner.",
         "condition": "R", "sense": 0},

        {"id": "bark_L_05", "sentence": "Silenced by a single hand signal from the handler, the bark stopped immediately.",
         "condition": "L", "sense": 0},
        {"id": "bark_R_05", "sentence": "The bark stopped immediately, silenced by a single hand signal from the handler.",
         "condition": "R", "sense": 0},

        # Sense 1: tree outer layer
        {"id": "bark_L_06", "sentence": "Rough and deeply furrowed from decades of growth, the bark of the oak was distinctive.",
         "condition": "L", "sense": 1},
        {"id": "bark_R_06", "sentence": "The bark of the oak was distinctive, rough and deeply furrowed from decades of growth.",
         "condition": "R", "sense": 1},

        {"id": "bark_L_07", "sentence": "Stripped away by deer rubbing their antlers in autumn, the bark was left pale and raw.",
         "condition": "L", "sense": 1},
        {"id": "bark_R_07", "sentence": "The bark was left pale and raw, stripped away by deer rubbing their antlers in autumn.",
         "condition": "R", "sense": 1},

        {"id": "bark_L_08", "sentence": "Used for centuries in traditional medicine to reduce fever, the bark was carefully harvested.",
         "condition": "L", "sense": 1},
        {"id": "bark_R_08", "sentence": "The bark was carefully harvested, used for centuries in traditional medicine to reduce fever.",
         "condition": "R", "sense": 1},

        {"id": "bark_L_09", "sentence": "Covered in bright green lichen and soft moss, the bark caught the morning light.",
         "condition": "L", "sense": 1},
        {"id": "bark_R_09", "sentence": "The bark caught the morning light, covered in bright green lichen and soft moss.",
         "condition": "R", "sense": 1},

        {"id": "bark_L_10", "sentence": "Peeling off in large papery sheets as the tree matured, the bark was pale silver.",
         "condition": "L", "sense": 1},
        {"id": "bark_R_10", "sentence": "The bark was pale silver, peeling off in large papery sheets as the tree matured.",
         "condition": "R", "sense": 1},
    ],

    "crane": [
        # Sense 0: bird
        {"id": "crane_L_01", "sentence": "Standing motionless at the marsh's edge on one long leg, the crane watched the water.",
         "condition": "L", "sense": 0},
        {"id": "crane_R_01", "sentence": "The crane watched the water, standing motionless at the marsh's edge on one long leg.",
         "condition": "R", "sense": 0},

        {"id": "crane_L_02", "sentence": "Migrating thousands of kilometres from Siberia to winter wetlands, the crane passed overhead.",
         "condition": "L", "sense": 0},
        {"id": "crane_R_02", "sentence": "The crane passed overhead, migrating thousands of kilometres from Siberia to winter wetlands.",
         "condition": "R", "sense": 0},

        {"id": "crane_L_03", "sentence": "Performing its elaborate courtship dance in the flooded meadow, the crane spread its wings.",
         "condition": "L", "sense": 0},
        {"id": "crane_R_03", "sentence": "The crane spread its wings, performing its elaborate courtship dance in the flooded meadow.",
         "condition": "R", "sense": 0},

        {"id": "crane_L_04", "sentence": "Spotted by birdwatchers in the reed beds of the wetland reserve, the crane was rare.",
         "condition": "L", "sense": 0},
        {"id": "crane_R_04", "sentence": "The crane was rare, spotted by birdwatchers in the reed beds of the wetland reserve.",
         "condition": "R", "sense": 0},

        {"id": "crane_L_05", "sentence": "Calling out with its distinctive bugling sound across the open farmland, the crane lifted into flight.",
         "condition": "L", "sense": 0},
        {"id": "crane_R_05", "sentence": "The crane lifted into flight, calling out with its distinctive bugling sound across the open farmland.",
         "condition": "R", "sense": 0},

        # Sense 1: construction machine
        {"id": "crane_L_06", "sentence": "Operated from a heated cab sixty metres above the street, the crane swung the load.",
         "condition": "L", "sense": 1},
        {"id": "crane_R_06", "sentence": "The crane swung the load, operated from a heated cab sixty metres above the street.",
         "condition": "R", "sense": 1},

        {"id": "crane_L_07", "sentence": "After engineers calculated the maximum rated load, the crane lifted the steel beam.",
         "condition": "L", "sense": 1},
        {"id": "crane_R_07", "sentence": "The crane lifted the steel beam after engineers calculated the maximum rated load.",
         "condition": "R", "sense": 1},

        {"id": "crane_L_08", "sentence": "Deployed to extract the collapsed roof section of the building, the crane worked through the night.",
         "condition": "L", "sense": 1},
        {"id": "crane_R_08", "sentence": "The crane worked through the night, deployed to extract the collapsed roof section of the building.",
         "condition": "R", "sense": 1},

        {"id": "crane_L_09", "sentence": "Its cable inspected for fraying before each shift, the crane was rated for 40 tonnes.",
         "condition": "L", "sense": 1},
        {"id": "crane_R_09", "sentence": "The crane was rated for 40 tonnes, its cable inspected for fraying before each shift.",
         "condition": "R", "sense": 1},

        {"id": "crane_L_10", "sentence": "Folding onto the back of a flatbed truck for transport between sites, the crane was portable.",
         "condition": "L", "sense": 1},
        {"id": "crane_R_10", "sentence": "The crane was portable, folding onto the back of a flatbed truck for transport between sites.",
         "condition": "R", "sense": 1},
    ],

    # TODO: add entries for bark, spring, match, light, pitch
    # following the same pattern above.
}

# Quick stats
for word, items in PAIRED.items():
    L = sum(1 for i in items if i['condition'] == 'L')
    R = sum(1 for i in items if i['condition'] == 'R')
    print(f"{word:<8}: {L} L, {R} R")

## 3. Validate — homonym appears in every sentence

In [ ]:
issues = []
for word, items in PAIRED.items():
    for item in items:
        if word.lower() not in item['sentence'].lower():
            issues.append(f"  MISSING '{word}' in {item['id']}: {item['sentence'][:60]}")

if issues:
    print("Problems:")
    for msg in issues: print(msg)
else:
    print("All sentences contain the target word.")

## 4. Garden-path sentences (H5)

These are sentences where **left context strongly primes sense A**, but the **full sentence resolves to sense B**.  
Label both `primed_sense` and `correct_sense`.

In [ ]:
GARDEN_PATH = {
    "bank": [
        {
            "id":            "bank_gp_01",
            "sentence":      "She had been saving for months and finally made her way to the bank, its muddy slopes crumbling into the floodwater.",
            "primed_sense":  1,  # left context primes financial institution
            "correct_sense": 0,  # full sentence resolves to river bank
        },
        {
            "id":            "bank_gp_02",
            "sentence":      "He withdrew all his money and drove straight to the bank, where the heron stood knee-deep in water.",
            "primed_sense":  1,
            "correct_sense": 0,
        },
        {
            "id":            "bank_gp_03",
            "sentence":      "The mortgage papers were ready and she headed to the bank, which was eroding badly after the spring floods.",
            "primed_sense":  1,
            "correct_sense": 0,
        },
        {
            "id":            "bank_gp_04",
            "sentence":      "Knee-deep in mud, the otters slid down the bank and deposited their catch in the shallows.",
            "primed_sense":  0,  # left context primes river bank
            "correct_sense": 0,  # remains river bank — non-reversal control
        },
    ],
    "bat": [
        {
            "id":            "bat_gp_01",
            "sentence":      "He polished his bat carefully before the match and it swooped across the evening sky.",
            "primed_sense":  0,  # sports equipment
            "correct_sense": 1,  # flying mammal
        },
        {
            "id":            "bat_gp_02",
            "sentence":      "The cricket coach handed her the bat, which hung upside down from the cave ceiling.",
            "primed_sense":  0,
            "correct_sense": 1,
        },
    ],
    # TODO: add more garden-path items for bark, crane, and other words
}

for word, items in GARDEN_PATH.items():
    print(f"{word}: {len(items)} garden-path sentences")
    for item in items:
        print(f"  [{item['id']}] primed={item['primed_sense']} → correct={item['correct_sense']}")
        print(f"    {item['sentence'][:80]}")

## 5. Save both datasets

In [ ]:
import json
from pathlib import Path

# ╔══════════════════════════════════════════════════════════════════╗
# ║  Review all sentences above before running this cell.           ║
# ╚══════════════════════════════════════════════════════════════════╝

paired_path = Path("paired_sentences.json")
paired_path.write_text(json.dumps(PAIRED, indent=2, ensure_ascii=False))
print(f"Saved paired sentences → {paired_path}")

gp_path = Path("garden_path_sentences.json")
gp_path.write_text(json.dumps(GARDEN_PATH, indent=2, ensure_ascii=False))
print(f"Saved garden-path sentences → {gp_path}")

total_paired = sum(len(v) for v in PAIRED.values())
total_gp     = sum(len(v) for v in GARDEN_PATH.values())
print(f"\nTotal: {total_paired} paired sentences, {total_gp} garden-path sentences")